In [7]:
!unzip /content/drive/MyDrive/OMR-Datasets/OMR_5Fold_ROIs.zip

Streaming output truncated to the last 5000 lines.
  inflating: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam5_50_1_box9.jpg  
  inflating: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam2_67_1_box24.jpg  
  inflating: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam4_52_1_box6.jpg  
  inflating: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam5_83_1_box26.jpg  
  inflating: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam5_222_1_box15.jpg  
  inflating: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam5_47_1_box0.jpg  
  inflating: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam5_24_1_box40.jpg  
  inflating: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam5_128_1_box29.jpg  
  inflating: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam5_88_1_box36.jpg  
  inflating: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam5_190_1_box47.jpg  
  inflating: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam5_57_1_box19.jpg  
  inflating: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam5_251_1_box59.jpg  
  inflating: OMR_5Fold_ROIs/Fold_2/val/confirmed/exam2_40_1_box27.jpg  
  inflating:

In [3]:
!pip install Augmentor

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import Augmentor
import os
import glob
import shutil

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ
# ==========================================
# Đường dẫn gốc chứa 5 folds của ảnh ROIs (Ổ cứng SSD Colab)
K_FOLDS_ROIS_DIR = "/content/OMR_5Fold_ROIs"

# Số lượng ảnh mục tiêu bạn muốn có sau khi tăng cường cho mỗi Fold
TARGET_AUG_IMAGES = 1500

print(f"🚀 BẮT ĐẦU TĂNG CƯỜNG ẢNH CROSSED-OUT CHO 5 FOLDS\n")

for fold in range(1, 6):
    fold_name = f"Fold_{fold}"
    print(f"{'='*50}")
    print(f"📂 Đang xử lý {fold_name}...")

    # Chỉ trỏ vào thư mục crossedout của tập TRAIN (Quy tắc Zero-Leakage)
    train_crossedout_dir = os.path.join(K_FOLDS_ROIS_DIR, fold_name, "train", "crossedout")

    if not os.path.exists(train_crossedout_dir):
        print(f"❌ Không tìm thấy thư mục: {train_crossedout_dir}")
        continue

    # Đếm số lượng ảnh gốc hiện có
    original_files = glob.glob(os.path.join(train_crossedout_dir, "*.*"))
    num_original_images = len(original_files)
    images_to_generate = TARGET_AUG_IMAGES - num_original_images

    if images_to_generate <= 0:
        print(f"✅ Thư mục đã có đủ {num_original_images} ảnh, bỏ qua.")
        continue

    print(f"Có {num_original_images} ảnh gốc. Đang sinh thêm {images_to_generate} ảnh...")

    # ==========================================
    # KHỞI TẠO PIPELINE TĂNG CƯỜNG (GEOMETRIC AUGMENTATION)
    # ==========================================
    # Augmentor sẽ tự động quét ảnh trong thư mục và tạo thư mục con tên là 'output'
    p = Augmentor.Pipeline(train_crossedout_dir)

    # 1. Xoay nhẹ (Rotation) từ -12 đến 12 độ
    p.rotate(probability=0.7, max_left_rotation=12, max_right_rotation=12)

    # 2. Phóng to/Thu nhỏ nhẹ (Zoom) - Mô phỏng nét gạch to nhỏ khác nhau
    p.zoom(probability=0.5, min_factor=0.9, max_factor=1.1)

    # # 3. Lật ngang/dọc (Flip) - Rất tốt cho lớp gạch chéo (chữ X lật chiều nào cũng là gạch bỏ)
    # p.flip_left_right(probability=0.5)
    # p.flip_top_bottom(probability=0.5)

    # 3. Làm méo mờ nhẹ (Random Distortion) - Mô phỏng nếp nhăn giấy
    p.random_distortion(probability=0.3, grid_width=4, grid_height=4, magnitude=2)

    # 4. Dịch chuyển xô nghiêng (Shear) - Mô phỏng độ nghiêng của camera
    p.shear(probability=0.3, max_shear_left=5, max_shear_right=5)

    # Thực thi quá trình sinh ảnh
    p.sample(images_to_generate)

    # ==========================================
    # DỌN DẸP VÀ ĐỔI TÊN FILE
    # ==========================================
    output_dir = os.path.join(train_crossedout_dir, "output")
    aug_files = glob.glob(os.path.join(output_dir, "*.*"))

    print(f"Đang di chuyển {len(aug_files)} ảnh ra thư mục chính...")

    for idx, file_path in enumerate(aug_files):
        # Tạo tên file mới gọn gàng hơn, không bị dính mã hash dài dòng của Augmentor
        ext = os.path.splitext(file_path)[1]
        new_file_name = f"aug_{fold_name}_geom_{idx}{ext}"
        new_file_path = os.path.join(train_crossedout_dir, new_file_name)

        # Di chuyển file từ thư mục output ra thư mục train_crossedout_dir
        shutil.move(file_path, new_file_path)

    # Xóa thư mục output rỗng
    if os.path.exists(output_dir):
        os.rmdir(output_dir)

    # Kiểm tra lại tổng số lượng ảnh cuối cùng
    total_images_now = len(glob.glob(os.path.join(train_crossedout_dir, "*.*")))
    print(f"✅ Hoàn tất {fold_name}! Thư mục hiện có tổng cộng {total_images_now} ảnh.")

print(f"\n{'='*50}")
print("🎉 HOÀN TẤT TĂNG CƯỜNG DỮ LIỆU CƠ BẢN CHO CẢ 5 FOLDS!")
print("Bây giờ bạn đã có thể dùng dữ liệu này để bắt đầu huấn luyện DCGAN cho từng Fold.")

🚀 BẮT ĐẦU TĂNG CƯỜNG ẢNH CROSSED-OUT CHO 5 FOLDS

📂 Đang xử lý Fold_1...
Có 122 ảnh gốc. Đang sinh thêm 1378 ảnh...
Initialised with 122 image(s) found.
Output directory set to /content/OMR_5Fold_ROIs/Fold_1/train/crossedout/output.

Processing <PIL.Image.Image image mode=RGB size=86x86 at 0x7E0DD6E383E0>: 100%|██████████| 1378/1378 [00:05<00:00, 247.61 Samples/s]


Đang di chuyển 1378 ảnh ra thư mục chính...
✅ Hoàn tất Fold_1! Thư mục hiện có tổng cộng 1500 ảnh.
📂 Đang xử lý Fold_2...
Có 118 ảnh gốc. Đang sinh thêm 1382 ảnh...
Initialised with 118 image(s) found.
Output directory set to /content/OMR_5Fold_ROIs/Fold_2/train/crossedout/output.

Processing <PIL.Image.Image image mode=RGB size=137x76 at 0x7E0DD3B5C6E0>: 100%|██████████| 1382/1382 [00:08<00:00, 164.46 Samples/s]


Đang di chuyển 1382 ảnh ra thư mục chính...
✅ Hoàn tất Fold_2! Thư mục hiện có tổng cộng 1500 ảnh.
📂 Đang xử lý Fold_3...
Có 121 ảnh gốc. Đang sinh thêm 1379 ảnh...
Initialised with 121 image(s) found.
Output directory set to /content/OMR_5Fold_ROIs/Fold_3/train/crossedout/output.

...

Đang di chuyển 1379 ảnh ra thư mục chính...
✅ Hoàn tất Fold_3! Thư mục hiện có tổng cộng 1500 ảnh.
📂 Đang xử lý Fold_4...
Có 120 ảnh gốc. Đang sinh thêm 1380 ảnh...
Initialised with 120 image(s) found.
Output directory set to /content/OMR_5Fold_ROIs/Fold_4/train/crossedout/output.

Processing <PIL.Image.Image image mode=RGB size=83x82 at 0x7E0DD3A589E0>: 100%|██████████| 1380/1380 [00:07<00:00, 192.36 Samples/s]


Đang di chuyển 1380 ảnh ra thư mục chính...
✅ Hoàn tất Fold_4! Thư mục hiện có tổng cộng 1500 ảnh.
📂 Đang xử lý Fold_5...
Có 115 ảnh gốc. Đang sinh thêm 1385 ảnh...
Initialised with 115 image(s) found.
Output directory set to /content/OMR_5Fold_ROIs/Fold_5/train/crossedout/output.

Processing <PIL.Image.Image image mode=RGB size=80x82 at 0x7E0DD3C7A0C0>: 100%|██████████| 1385/1385 [00:06<00:00, 203.90 Samples/s]


Đang di chuyển 1385 ảnh ra thư mục chính...
✅ Hoàn tất Fold_5! Thư mục hiện có tổng cộng 1500 ảnh.

🎉 HOÀN TẤT TĂNG CƯỜNG DỮ LIỆU CƠ BẢN CHO CẢ 5 FOLDS!
Bây giờ bạn đã có thể dùng dữ liệu này để bắt đầu huấn luyện DCGAN cho từng Fold.


In [9]:
!zip -q -r /content/drive/MyDrive/OMR-Datasets/OMR_5Fold_ROIs_split.zip /content/OMR_5Fold_ROIs_split/

In [10]:
import os

len(os.listdir("/content/OMR_5Fold_ROIs_split/Fold_1/train/crossedout"))

1500